# 01 - Data Cleaning & Preprocessing

In this notebook, we prepare the flight booking dataset for machine learning.

The goal is to:
- Clean the dataset (missing values, duplicates)
- Understand feature types
- Encode categorical variables
- Scale numerical features
- Build a preprocessing pipeline to avoid data leakage

## Cleaning

In [1]:
# basic libraries
import pandas as pd
import numpy as np

In [2]:
# load dataset
df = pd.read_csv("../data/raw/flightData.csv")

# first look
df.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,NaN,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [3]:
# now explore the dataset to understand its structure and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        300153 non-null  int64  
 1   airline           300153 non-null  str    
 2   flight            300153 non-null  str    
 3   source_city       300153 non-null  str    
 4   departure_time    300153 non-null  str    
 5   stops             300153 non-null  str    
 6   arrival_time      300153 non-null  str    
 7   destination_city  300153 non-null  str    
 8   class             300153 non-null  str    
 9   duration          300149 non-null  float64
 10  days_left         300153 non-null  int64  
 11  price             300153 non-null  int64  
dtypes: float64(1), int64(3), str(8)
memory usage: 27.5 MB


In [4]:
df.describe()

,Unnamed: 0,duration,days_left,price
count,300153.000000,300149.000000,300153.000000,300153.000000
mean,150076.000000,12.221070,26.004751,20889.660523
std,86646.852011,7.192012,13.561004,22697.767366
min,0.000000,0.830000,1.000000,1105.000000
25%,75038.000000,6.830000,15.000000,4783.000000
50%,150076.000000,11.250000,26.000000,7425.000000
75%,225114.000000,16.170000,38.000000,42521.000000
max,300152.000000,49.830000,49.000000,123071.000000


In [5]:
df.columns

Index(['Unnamed: 0', 'airline', 'flight', 'source_city', 'departure_time',
       'stops', 'arrival_time', 'destination_city', 'class', 'duration',
       'days_left', 'price'],
      dtype='str')

In [6]:
# this column is an index column saved inside the dataset file.
# it does not contain any meaningful information for prediction, so we remove it.
df = df.drop(columns=["Unnamed: 0"])

In [7]:
# check missing values
print("Missing values before cleaning:")
df.isnull().sum()

Missing values before cleaning:


airline             0
flight              0
source_city         0
departure_time      0
stops               0
arrival_time        0
destination_city    0
class               0
duration            4
days_left           0
price               0
dtype: int64

In [8]:
# missing values are removed because the dataset is large, so removing a small number of rows does not significantly affect the analysis.
df = df.dropna()

# check missing values again
print("Missing values after cleaning:")
print(df.isnull().sum())


Missing values after cleaning:
airline             0
flight              0
source_city         0
departure_time      0
stops               0
arrival_time        0
destination_city    0
class               0
duration            0
days_left           0
price               0
dtype: int64


In [9]:
# check number of duplicate rows
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicate rows removed.")
else : 
    print("No duplicate rows found.")

Number of duplicate rows: 0
No duplicate rows found.


## Data Verification

We verify that the dataset structure matches the project description by checking its columns and feature consistency.

In [10]:
# we verify the validity of the airline feature by checking its structure and consistency with the project description.
# we expect to find 6 unique airlines based on the project description

unique_airlines = df["airline"].unique()

print("Airlines found in dataset:", unique_airlines)
print("Number of unique airlines:", len(unique_airlines))

if len(unique_airlines) == 6:
    print("Airline feature is valid (6 airlines found).")
else:
    print("Warning: Airline feature does not match expected structure.")

Airlines found in dataset: <StringArray>
['SpiceJet', 'AirAsia', 'Vistara', 'GO_FIRST', 'Indigo', 'Air_India']
Length: 6, dtype: str
Number of unique airlines: 6
Airline feature is valid (6 airlines found).


In [11]:
# source & destination city
# we expect to find 6 unique source cities and 6 unique destination cities

unique_source_cities = df["source_city"].unique()

print("Source cities found in dataset:", unique_source_cities)
print("Number of unique source cities:", len(unique_source_cities))

if len(unique_source_cities) == 6:
    print("Source city feature is valid (6 cities found).")
else:
    print("Warning: Source city feature does not match expected structure.")


unique_destination_cities = df["destination_city"].unique()
print("Destination cities found in dataset:", unique_destination_cities)
print("Number of unique destination cities:", len(unique_destination_cities))

if len(unique_destination_cities) == 6:
    print("Destination city feature is valid (6 cities found).")
else:
    print("Warning: Destination city feature does not match expected structure.")

Source cities found in dataset: <StringArray>
['Delhi', 'Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai']
Length: 6, dtype: str
Number of unique source cities: 6
Source city feature is valid (6 cities found).
Destination cities found in dataset: <StringArray>
['Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai', 'Delhi']
Length: 6, dtype: str
Number of unique destination cities: 6
Destination city feature is valid (6 cities found).


In [12]:
# departure and arrival time
# we expect to find 6 unique departure time categories and 6 unique arrival time categories

unique_departure_times = df["departure_time"].unique()
departure_count = len(unique_departure_times)

print("Departure times found in dataset:", unique_departure_times)
print("Number of departure time categories:", departure_count)

if departure_count == 6:
    print("Departure time feature is valid (6 categories found).")
else:
    print("Warning: Departure time feature does not match expected structure.")



unique_arrival_times = df["arrival_time"].unique()
arrival_count = len(unique_arrival_times)

print("Arrival times found in dataset:", unique_arrival_times)
print("Number of arrival time categories:", arrival_count)

if arrival_count == 6:
    print("Arrival time feature is valid (6 categories found).")
else:
    print("Warning: Arrival time feature does not match expected structure.")

Departure times found in dataset: <StringArray>
['Evening', 'Early_Morning', 'Morning', 'Afternoon', 'Night', 'Late_Night']
Length: 6, dtype: str
Number of departure time categories: 6
Departure time feature is valid (6 categories found).
Arrival times found in dataset: <StringArray>
['Night', 'Early_Morning', 'Afternoon', 'Morning', 'Evening', 'Late_Night']
Length: 6, dtype: str
Number of arrival time categories: 6
Arrival time feature is valid (6 categories found).


In [13]:
# stops
# we expect to find 3 unique stop categories based on the project description (non-stop, 1 stop, 2+ stops)

unique_stops = df["stops"].unique()

print("Stops found in dataset:", unique_stops)
print("Number of unique stop categories:", len(unique_stops))

if len(unique_stops) == 3:
    print("Stops feature is valid (3 categories found).")
else:
    print("Warning: Stops feature does not match expected structure.")

Stops found in dataset: <StringArray>
['zero', 'one', 'two_or_more']
Length: 3, dtype: str
Number of unique stop categories: 3
Stops feature is valid (3 categories found).


In [14]:
# Class
# we expect to find 2 unique class categories (economy, business)

unique_classes = df["class"].unique()

print("Class categories found in dataset:", unique_classes)
print(f"Number of class categories: {len(unique_classes)}")

if len(unique_classes) == 2:
    print("Class feature is valid.")
else:
    print("Warning: Unexpected number of class categories.")

Class categories found in dataset: <StringArray>
['Economy', 'Business']
Length: 2, dtype: str
Number of class categories: 2
Class feature is valid.


In [15]:
print("Final dataset shape:", df.shape)
print("First few rows of the cleaned dataset:")
df.head()

Final dataset shape: (300149, 11)
First few rows of the cleaned dataset:


,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955
5,Vistara,UK-945,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.33,1,5955


In [16]:
# we save the cleaned datasets.

df.to_csv("../data/cleaned/flight_data.csv", index=False)

print("Dataset cleaning completed successfully.")
print("Dataset is ready for EDA and modeling.")

Dataset cleaning completed successfully.
Dataset is ready for EDA and modeling.
